# Regresja logistyczna

W tym ćwiczeniu zbudujemy klasyfikator oparty na regresji logistycznej. 
Rozważymy klasyczny problem klasyfikacji odmian irysów na podstawie cech płatków. Jest to klasyczny już problem, często wykorzystywany przy porównywaniu różnych technik klasyfikacji. Więcej o pochodzeniu tych danych i problemie można przeczytać tu: [Iris_flower_data](https://en.wikipedia.org/wiki/Iris_flower_data_set)

### Przygotowanie środowiska programistycznego

In [20]:
import pandas as pd
import numpy as np
import scipy.optimize as so
import plotly.express as px

from sklearn import datasets


In [21]:
iris = datasets.load_iris() 
df = pd.DataFrame(iris.data, columns=iris.feature_names)
df["classes"] = iris.target
print(df.describe())
print(df.head())

       sepal length (cm)  sepal width (cm)  petal length (cm)  \
count         150.000000        150.000000         150.000000   
mean            5.843333          3.057333           3.758000   
std             0.828066          0.435866           1.765298   
min             4.300000          2.000000           1.000000   
25%             5.100000          2.800000           1.600000   
50%             5.800000          3.000000           4.350000   
75%             6.400000          3.300000           5.100000   
max             7.900000          4.400000           6.900000   

       petal width (cm)     classes  
count        150.000000  150.000000  
mean           1.199333    1.000000  
std            0.762238    0.819232  
min            0.100000    0.000000  
25%            0.300000    0.000000  
50%            1.300000    1.000000  
75%            1.800000    2.000000  
max            2.500000    2.000000  
   sepal length (cm)  sepal width (cm)  petal length (cm)  petal width (

## Pre-processing, oraz analiza wizualna danych. 

Najpierw przygotujemy uproszczoną wersję danych do klasyfikacji i nazwijmy to df_sub.
**Proszę:**
* wybrać kolumny ```'petal length (cm)', 'petal width (cm)', 'classes'```
* dla ułatwienia - zmienić nazwy kolumn na ```'length' i 'width'```
* stworzyć kolumnę ```label```, która będzie miałą następujące wartości: 0 dla classes 0 lub 1 oraz wartość 1 dla classes=2.
* usunąć powtarzające się wiersze




In [22]:
#BEGIN_SOLUTION
df_sub = df[['petal length (cm)', 'petal width (cm)', 'classes']]
df_sub = df_sub.rename(columns={"petal width (cm)": "width", "petal length (cm)": "length"})
df_sub['label'] = df_sub['classes'].map({0: 0, 1: 0, 2: 1})
df_sub = df_sub.drop_duplicates()
#END_SOLUTION
df_sub

,length,width,classes,label
0,1.4,0.2,0,0
2,1.3,0.2,0,0
3,1.5,0.2,0,0
5,1.7,0.4,0,0
6,1.4,0.3,0,0
...,...,...,...,...
145,5.2,2.3,2,1
146,5.0,1.9,2,1
147,5.2,2.0,2,1
148,5.4,2.3,2,1



Teraz przejdziemy do analizy danych. Korzystając z metod klasy DataFrame.

**Proszę:**

* narysować histogramy cech (width, length) w zależności od kolumny label (osobne kolory)
* zrób wykres px.scatter, gdzie x to width, y to length, color to label z wykorzystaniem parametrów marginal_x i marginal_y dla różnych wariantów (box, histogram itp.) i ostatecznie wybierz jeden wariant.
**Wskazówka** Stwórz kolumnę ``` df_sub["Survived_str"] = df_sub["Survived"].astype(str) ``` aby plotly uznało to za kategorię.

In [23]:

#BEGIN_SOLUTION
px.histogram(df_sub, x="width", nbins=30, color='label', title="width distribution", opacity=0.7).show()
px.histogram(df_sub, x="length", nbins=30, color='label', title="length distribution", opacity=0.7).show()
#END_SOLUTION


In [24]:

df_sub["label_str"] = df_sub["label"].astype(str)
px.scatter(
    df_sub, 
    x="width", 
    y="length", 
    color="label_str",
    marginal_x="histogram",   # inne opcje: "box", "violin", "rug"
    marginal_y="histogram"
)

## Hipoteza
Dla przypomnienia _hipoteza_ w regresji logistycznej ma postać: 

$\qquad$ $h_\theta(x) = \frac{1}{1+\exp(-\theta x^T )}$.

W implementacji dobrze jest myśleć o tej funkcji tak:

$\qquad$ $h_\theta(x) = \frac{1}{1+f}$.

gdzie: $f = \exp(-\theta x^T)$

**Proszę** napisać funkcję ```logistic_func(x, theta)``` która:

* implementuje funkcję logistyczną
* jako argumenty przyjmuje parametry regresji logistycznej  $(\theta_{0}, \theta_{1}, ..., \theta_{i})$ oraz tablicę danych wejściowych $x$. 
* w kodzie funkcji proszę rozszerzyć tablicę $x$ o dodatkową kolumnę jedynek, by parametr $\theta_{0}$ był traktowany na tej same zasadzie co pozostałe parametry
* ze względu na stabilność numeryczną obliczeń proszę ograniczyć wartości wykładnika w mianowniku do zakresu  $\pm18$

**Ostrzeżenie:** x to tablica która może zawierać wiele kolumn i wiele wierszy.

**Wskazówka**: ograniczając zakres zwracanych wartości proszę skorzystać z funkcji ```np.where()``` zaaplikowanej do wektora wartości wykładnika.

Proszę sprawdzić działanie funkcji na następujących danych testowych:
```Python
theta = np.array([1,1,2])
x = np.array([[5,5],
              [5,6],
              [-5,-5],
              [-5,-8]])
```
Oczekiwany wynik:
```
[9.99999887e-01 9.99999985e-01 8.31528028e-07 1.52299795e-08]
```

In [25]:
def logistic_func(theta, x):
    # dodaj kolumne jedynek
    #BEGIN_SOLUTION
    x_expanded = np.column_stack((np.ones(x.shape[0]), x))
    #arg1 = (theta.reshape(1, len(theta)) @ x_expanded.T).reshape(-1)
    #END_SOLUTION
    # policz argument funkcji
    #BEGIN_SOLUTION
    arg = np.sum(theta*x_expanded, axis=1)
    #END_SOLUTION
    # uzyj np.where żeby ograniczyc wartosci parmetru do [-18,18]
    #BEGIN_SOLUTION
    arg = np.where(np.abs(arg)<18, arg, 18*np.sign(arg))
    #END_SOLUTION
    return 1.0/(1+np.exp(-arg))

theta = np.array([1,1,2])
x = np.array([[5,5],
              [5,6],
              [-5,-5],
              [-5,-8]])
res = logistic_func(theta, x)

# sprawdzenie poprawności rozmiaru listy
assert res.shape == (4,)
print(res)

[9.99999887e-01 9.99999985e-01 8.31528028e-07 1.52299795e-08]


## Funkcja log-wiarygodności (LLH): 
Parametry regresji znajdujemy przez maksymalizację [funkcji log-wiarygodności](https://brain.fuw.edu.pl/edu/index.php/Uczenie_maszynowe_i_sztuczne_sieci_neuronowe/Wykład_6#Funkcja_wiarygodno.C5.9Bci):

$\qquad$ $l(\theta) = \log L(\theta) = \sum_{j=1}^m y^{(j)} \log h(x^{(j)}) + (1 - y^{(j)}) \log (1 - h(x^{(j)}))$,
gdzie:  

m - liczebność próbki

x - dane wejściowe, u nas length, width

y - dane wyjściowe, u nas label

h - postać zależności wyniku od danych wejściowych. U nas to jest funkcja logistyczna, czyli oczekujemy, że wzór y = h(x) dobrze opisuje zależność między danymi wejściowymi, a wyjściowymi.


<hr>

**Proszę** napisać funkcję ```log_likelihood(theta, x,y, model)``` która:

* implementuje funkcję log-wiarygodności
* jako argumenty przyjmuje parametry regresji logistycznej  $(\theta_{0}, \theta_{1}, ..., \theta_{i})$ oraz tablicę danych wejściowych $x, y$. 
* ```model``` dla którego szukamy parametrów $\theta_{i}$ w naszym przypadku to będzie funkcja logistyczna: ```logistic_func```

**Uwaga**: argument $theta$ musi być pierwszy 

In [26]:
def log_likelihood(theta, x, y, model):
    #BEGIN_SOLUTION
    model_result = model(theta, x)
    result = y*np.log(model_result) + (1-y)*np.log(1-model_result)
    result = np.sum(result)
    #END_SOLUTION
    return result

Maksymalizacja to zadanie optymalizacyjne - szukamy optymalnych parametrów, a kryterium optymalności to maksymalna wartość funkcji log-wiarygodności.
W tym ćwiczeniu zrobimy to za pomocą funkcji optymalizacyjnych z modułu [<tt>scipy.optimize</tt>]( http://docs.scipy.org/doc/scipy/reference/optimize.html#module-scipy.optimize). 


Wynikają z tego dwie konsekwencje:
* funkcje te są przystosowane do szukania minimów funkcji celu (straty). Musimy więc podawać im jako argumenty funkcję minus log-wiarygodności
* niektóre algorytmy mogą działać szybciej jeśli zaimplementujemy jawnie postać pochodnej:

$\qquad$ $
\begin{array}{lcl}
\frac{\partial}{\partial \theta_i} l(\theta)  =\sum_{j=1}^m (y^{(j)}-h_\theta(x^{(j)}))x_i^{(j)}
\end{array}
$

**Proszę** napisać funkcję ```negative_log_likelihood(theta, x,y, model)``` która:

* zwraca funkcję log-wiarygodności pomnożoną przez $-1$

In [27]:
def negative_log_likelihood(theta, x, y, model):
    #BEGIN_SOLUTION
    return -log_likelihood(theta, x, y, model)
    #END_SOLUTION 

**Proszę** napisać funkcję ```log_likelihood_derivative(theta, x,y, model)``` oraz ```negative_log_likelihood_derivative(theta, x,y, model)``` które:

* zwraca funkcję pochodną log-wiarygodności
* zwraca funkcję pochodną log-wiarygodności pomnożoną przez $-1$

**Uwaga**: mnożąc przez $x_{i}$ trzeba uwzględnić kolumnę jedynek

In [28]:
def log_likelihood_derivative(theta, x, y, model):
    # odpowiedź modelu na dane
    #BEGIN_SOLUTION
    x_expanded = np.column_stack((np.ones(x.shape[0]), x))
    model_result = model(theta, x)
    #END_SOLUTION
    # różnica odpowiedzi modelu i wartości prawdziwej
    #BEGIN_SOLUTION
    delta = np.array(y - model_result)
    #END_SOLUTION
    # obliczenie pochodnej wysumowanej po wszyskich przykładach
    #BEGIN_SOLUTION
    delta = np.reshape(delta,(-1,1))
    result = delta*x_expanded
    result = np.sum(result, axis=0)
    #END_SOLUTION
    #sprawdzenie poprawności rozmiaru wyniku
    assert result.shape == theta.shape
    return result

def negative_log_likelihood_derivative(theta, x, y, model):
    return -log_likelihood_derivative(theta, x, y, model)

## Procedura minimalizacji funkcji log-wiarygodności ze wsględu na parametry $\theta$ dla konkretnych danych.

W naszym przypadku mamy trzy parametry $\theta$ - mnożące odpowiednio 1, oraz kolumny Age i Fare.

**Proszę:**
* zainicjalizować parametry $\theta_{0}, \theta_{1}, \theta_{2}$ na wartości (0,0,0).
* obliczyć wartość i pochodną funkcji wiarygodności na danych, ```df```

Poprawny wynik to:
```Python
Wartość funkcji log-wiarygodności dla zbioru testowego = -71.39415959767437
Wartość pochodnej funkcji log-wiarygodności dla zbioru testowego = [-6.5  32.65 18.95]
```

In [29]:
#BEGIN_SOLUTION
theta0 = np.array([0,0,0])
model = logistic_func
X = df_sub[["length", "width"]]
y = df_sub["label"]
#END_SOLUTION

# wartość funkcji log-wiarygodności
#BEGIN_SOLUTION
llh = log_likelihood(theta0, X, y, model)
#END_SOLUTION
# wartość pochodnej
#BEGIN_SOLUTION
llh_derivative = log_likelihood_derivative(theta0, X, y, model)
#END_SOLUTION

print("Wartość funkcji log-wiarygodności dla zbioru testowego = {}".format(llh))
print("Wartość pochodnej funkcji log-wiarygodności dla zbioru testowego = {}".format(llh_derivative))

Wartość funkcji log-wiarygodności dla zbioru testowego = -71.39415959767437
Wartość pochodnej funkcji log-wiarygodności dla zbioru testowego = [-6.5  32.65 18.95]


## Optymalizacja  

Funkcje optymalizujące zaczerpniemy z modułu scipy.optimize: ```scipy.optimize.fmin_bfgs```. Ponieważ funkcje te są zaimplementowane do mnimalizowania to zamiast maksymalizować funkcję log-wiarygodności będziemy minimalizować funkcję przemnożoną przez -1 czyli ```f=negative_log_likelihood``` oraz ```fprime=negative_log_likelihood_derivative```

**Proszę:**

* wywołać funckję ```scipy.optimize.fmin_bfgs``` z odpowiednimi argumentami.
* porównać liczbę wywołań i czas wykonywania komórki z i bez podania explicite postaci pochodnej
(https://ipython.readthedocs.io/en/stable/interactive/magics.html?highlight=%25time#cell-magics)

Oczekiwany wynik:
```
Optymalne wartości parametrów theta: [-40.55228421   5.41888077   8.53935285]
Wartość funkcji log-wiarygodności dla optymalnych parametrów: -9.340587741669628
```

In [30]:
%%time 
# komenda ``time``` włącza licznik czasu wykonywania komórki

model = logistic_func

# poszukiwanie optymalnych wartości parametrów theta
#BEGIN_SOLUTION
theta_opt = so.fmin_bfgs(f=negative_log_likelihood, x0=theta0, 
                         #fprime=negative_log_likelihood_derivative, 
                         args=(X, y, model), disp= True)
#END_SOLUTION
# wartość LLH z optymalnymi parametrami
#BEGIN_SOLUTION
llh = log_likelihood(theta_opt, X, y, model)
#END_SOLUTION

print('Optymalne wartości parametrów theta: {}'.format(theta_opt))
print("Wartość funkcji log-wiarygodności dla optymalnych parametrów: {}".format(llh))

Optimization terminated successfully.
         Current function value: 9.340588
         Iterations: 25
         Function evaluations: 112
         Gradient evaluations: 28
Optymalne wartości parametrów theta: [-40.55228421   5.41888077   8.53935285]
Wartość funkcji log-wiarygodności dla optymalnych parametrów: -9.340587741669628
CPU times: user 74.5 ms, sys: 2.49 ms, total: 77 ms
Wall time: 79.1 ms


## Wyniki modelu
Wyniki regresji logistycznej możemy odbierać na dwa sposoby:
* obliczyć wartość hipotezy dla badanego wejścia i dopasowanych parametrów: miara ta ma interpretację prawdopodobieństwa przynależności wejścia do klasy 1,
* dopisać funkcję wykonującą klasyfikację, tzn. porównanie wartości hipotezy z 1/2: 
  * dla wartości hipotezy > 1/2 klasyfikacja zwraca 1, 
  * w przeciwnym razie 0.
  
  
**Proszę** napisać funkcję ```classification(theta, x, model)```  która:
* jako argument przyjmuje wektor parametrów modelu $\theta$, tablicę danych wejściowych $x$, oraz $model$
* zwraca listę klasyfikacyjną: $1$ gdy $model(x)>0.5$, a $0$ w przeciwnym przypadku

In [31]:
def classification(theta, x, model):
    #BEGIN_SOLUTION
    model_result = model(theta, x)
    #END_SOLUTION
    return model_result>0.5

In [32]:
df_sub

,length,width,classes,label,label_str
0,1.4,0.2,0,0,0
2,1.3,0.2,0,0,0
3,1.5,0.2,0,0,0
5,1.7,0.4,0,0,0
6,1.4,0.3,0,0,0
...,...,...,...,...,...
145,5.2,2.3,2,1,1
146,5.0,1.9,2,1,1
147,5.2,2.0,2,1,1
148,5.4,2.3,2,1,1


## Procedura klasyfikacji

**Proszę:**

* korzystając z modelu ```logistic_func``` wraz z parametrami zwróconymi przez procedurę optymalizacyjną obliczyć prawdopodobieństwo przynależności do klasy 1 dla wartości length=1.5 cm i width=3.0.

* korzystając z funkcji "classification" wyznaczyć czy kwiat należy do   klasy $0$ czy $1$.

Oczekiwany wynik:
```
Przynależność do klasy 1 wynosi: 0.0011
Kwiat zalicza się do klasy : [0]
```

In [38]:
# syntetyczne dane odpowiadające osobie z polecenia
#BEGIN_SOLUTION
x = np.array([[1.5, 3]])
p = model(theta_opt, x)
class_number = classification(theta_opt, x, model).astype(int)
#END_SOLUTION

print(f"Przynależność do klasy 1 wynosi: {p[0]:.4f}")
print(f"Kwiat zalicza się do klasy : {class_number}")

Przynależność do klasy 1 wynosi: 0.0011
Kwiat zalicza się do klasy : [0]


**Proszę:** 

* narysować punkty pokolorowane zgodnie z przynależnością do klas
* dorysować prostą rozgraniczającą obszary "1" od "0".   Prosta ma ona równanie 

$\qquad$ $h_\theta(x)=1/2$, 

tzn:

$\qquad$ $\theta x^T = 0$

czyli 

$\theta_0 +\theta_1 x_1 + \theta_2 x_2 =0 $

Przekształcając to do równania prostej we współrzędnych $(x_1,x_2)$ mamy:

$- \theta_2 x_2 = \theta_0 +\theta_1 x_1 $

$ x_2 = - \frac{1}{\theta_2}( \theta_0 +\theta_1 x_1 )$

In [39]:
import plotly.express as px
import plotly.graph_objects as go
import numpy as np
#BEGIN_SOLUTION
# wykres punktowy z kolorami w zależności od wyniku
fig = px.scatter(
    df_sub,
    x="length",
    y="width",
    color="label_str", 
    color_discrete_map={"1": "blue", "0": "red"},
    labels={"color": "label"},
    title="Regresja logistyczna – podział klas"
)

# prosta rozdzielająca klasy
x_vals = np.array([df_sub["length"].min(), df_sub["length"].max()])
y_vals = -(theta_opt[0] + theta_opt[1]*x_vals) / theta_opt[2]

fig.add_trace(
    go.Scatter(
        x=x_vals, y=y_vals,
        mode="lines",
        name="logistic regression model",
        line=dict(color="black")
    )
)

fig.show()
#END_SOLUTION

Wariant rysunku z użyciem biblioteki seaborn

# Zadanie domowe

Zastosowanie regresji liniowej do innego rodzaju danych.

**Proszę:**

* wygenerować syntetyczne wyniki 100 egzaminów z biologi i matematyki według rozkładu płaskiego w zakresie ```[0,100]``` punktów
* proszę wygenerować efekt rekrutacji na studia według wzoru:

$$
\text{zdał} =\sqrt{(\text{wynik z matematyki}-50)^{2} + (\text{wynik z biologii}-50)^{2}}>25
$$
  

* narysować rozkład osób które zdały/nie zdały na płaszczyźnie ```x="matematyka", y="biologia"```
* przeprowadzić trening regresji logistycznej dla tych danych
* narysować krzywą rozdzielającą kategorie
* odpowiedzieć na pytanie **"Czy krzywa poprawnie rozdziela kategorie?"**

In [40]:
nPoints = 1000
#BEGIN_SOLUTION
x = 100*np.random.random_sample(nPoints)
y = 100*np.random.random_sample(nPoints)

df = pd.DataFrame(data=x, columns=["matematyka"])
df["biologia"] = y
df["wynik"] = np.sqrt((x-50)**2 + (y-50)**2)>25


model = logistic_func
theta0 = [0,0,0]
theta_opt = so.fmin_bfgs(f=negative_log_likelihood, x0=theta0, 
                         fprime=negative_log_likelihood_derivative, 
                         args=(df[["matematyka","biologia"]], df["wynik"], model), disp= True)
print('Optymalne wartości parametrów theta: {}'.format(theta_opt))


# wykres punktowy z kolorami w zależności od wyniku
fig = px.scatter(
    df,
    x="matematyka",
    y="biologia",
    color=df['wynik'].astype(str),  # traktujemy wynik jako kategorię
    color_discrete_map={"1": "blue", "0": "red"},
    labels={"color": "wynik"},
    title="Regresja logistyczna – podział klas"
)

# prosta rozdzielająca klasy

x_vals = np.array([df["matematyka"].min(),df["matematyka"].max()])
y_vals = -(theta_opt[0] + theta_opt[1]*x)/theta_opt[2]

fig.add_trace(
    go.Scatter(
        x=x_vals, y=y_vals,
        mode="lines",
        name="logistic regression model",
        line=dict(color="black")
    )
)

fig.show()

#END_SOLUTION


Optimization terminated successfully.
         Current function value: 482.899594
         Iterations: 12
         Function evaluations: 20
         Gradient evaluations: 20
Optymalne wartości parametrów theta: [ 1.43453979 -0.00146826  0.00204932]
